# Match two AnnData objects (1:1)

Matches objects from AnnData 1 to AnnData 2 by nearest centroid within each (Barcode, Well).
Creates a merged AnnData containing:
- obs from AD1 + prefixed obs from AD2 + match metadata
- features (X/var) from AD1 and AD2 with user-defined prefixes


# Imports

In [ ]:
import os

from Functions.Fcts_Base import load_adata
from Functions.Fcts_Matching import (
    match_1to1_by_centroid,
    merge_anndata_by_matches,
    save_matching_outputs,
)

%load_ext autoreload
%autoreload 2


# User Input

In [ ]:
# Paths to the two AnnData files produced by 1_FeatureExtraction (separately)
path_ad1 = "YOUR PATH TO AD1"  # e.g. cells
path_ad2 = "YOUR PATH TO AD2"  # e.g. nuclei

max_distance_um = 25.0 # Maximum allowed centroid distance for a match (micrometers)

# Feature prefixes (will be applied to var_names before merging)
prefix_ad1 = "cells"
prefix_ad2 = "nuclei"

# Load

In [ ]:
# ── Run Cell ───────────────────────────────────────────────────────────────────────
ad1 = load_adata(path_ad1, load_zarrs = False)
ad2 = load_adata(path_ad2, load_zarrs = False)

# Match, merge, save

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
# OBS naming. Change if your ADs use different column names for these features.
barcode_col = "Barcode"
well_col = "Well"
x_col = "centroid_x_micrometer"
y_col = "centroid_y_micrometer"

out_basename = "1b_Matched" # Result file naming

error_on_unmatched = True # If True: create merged output but raise ValueError if any AD1 objects are unmatched

# ── Run Cell ───────────────────────────────────────────────────────────────────────
prefix_ad2_obs = prefix_ad2

out_dir = os.path.dirname(path_ad1) # Output directory (where the merged .h5ad + reports are written)

match_df, unmatched = match_1to1_by_centroid(
    ad1=ad1,
    ad2=ad2,
    max_distance_um=max_distance_um,
    barcode_col=barcode_col,
    well_col=well_col,
    x_col=x_col,
    y_col=y_col,
)

matching_uns = {
    "path_ad1": path_ad1,
    "path_ad2": path_ad2,
    "max_distance_um": float(max_distance_um),
    "prefix_ad1": prefix_ad1,
    "prefix_ad2": prefix_ad2,
    "prefix_ad2_obs": prefix_ad2_obs,
    "barcode_col": barcode_col,
    "well_col": well_col,
    "x_col": x_col,
    "y_col": y_col,
    "n_ad1_total": int(ad1.n_obs),
    "n_ad2_total": int(ad2.n_obs),
    "n_matched": int(len(match_df)),
    "n_unmatched_ad1": int(len(unmatched)),
}

ad_merged = merge_anndata_by_matches(
    ad1=ad1,
    ad2=ad2,
    match_df=match_df,
    prefix_ad1=prefix_ad1,
    prefix_ad2=prefix_ad2,
    prefix_ad2_obs=prefix_ad2_obs,
    keep_uns_from="ad1",
    matching_uns=matching_uns,
)

save_matching_outputs(
    ad_merged=ad_merged,
    unmatched_ad1=unmatched,
    out_dir=out_dir,
    out_basename=out_basename,
    error_on_unmatched=error_on_unmatched,
)